# 1. Import and Install Dependencies

!pip install tensorflow==2.16.1 opencv-python mediapipe scikit-learn matplotlib

In [3]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

os.environ['KMP_DUPLICATE_LIB_OK']='True'

2025-07-30 20:29:08.119597: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# 2. Keypoints using MP Holistic

In [5]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [6]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [6]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

In [7]:
def draw_styled_landmarks(image, results):
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

# 3. Extract Keypoint Values

In [9]:
def extract_keypoints(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([lh, rh])

# 4. Setup Folders for Collection

In [11]:
# Path for exported data, numpy arrays
DATA_PATH = os.path.join('MP_Data') 

# Actions that we try to detect
actions = np.array(['Vsih vitay', 'Diakuy vam'])
# actions = np.array(['Vsih vitay', 'Diakuy vam', 'Byd laska', 'Povitriana tryvoga', 'Chi potribna dopomoga?'])

# Thirty videos worth of data
no_sequences = 50

# Videos are going to be 30 frames in length
sequence_length = 30

In [12]:
for action in actions: 
    for sequence in range(no_sequences):
        try: 
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

# 5. Collect Keypoint Values for Training and Testing

In [ ]:
import cv2
import time
import os
import logging
import mediapipe as mp
import numpy as np

logging.getLogger('mediapipe').setLevel(logging.ERROR)

# Suppress all non-critical logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GLOG_minloglevel'] = '3'

# MediaPipe setup
mp_holistic = mp.solutions.holistic

# Define missing variables
actions = ['hello', 'thanks', 'iloveyou']  # Your action classes
no_sequences = 30  # Number of sequences per action
sequence_length = 30  # Number of frames per sequence
DATA_PATH = os.path.join('MP_Data')  # Path to save data

def safe_read(cap):
    """Safely read a frame from video capture"""
    try:
        ret, frame = cap.read()
        return ret, frame
    except Exception as e:
        print(f"Error reading frame: {e}")
        return False, None

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = model.process(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    # Add your landmark drawing code here
    pass

def extract_keypoints(results):
    # Add your keypoint extraction code here
    return np.zeros(1662)  # Placeholder

# Create directories for data first
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

cap = cv2.VideoCapture(0)  # Try without CAP_AVFOUNDATION
if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

# Test window first
ret, test_frame = cap.read()
if ret:
    cv2.imshow('Test Window', test_frame)
    print("Test window created. Press any key to continue...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

time.sleep(1)

# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    # NEW LOOP
    # Loop through actions
    for action in actions:
        for sequence in range(no_sequences):
            print(f"Starting {action} sequence {sequence}. Press any key...")
            cv2.waitKey(0)
            
            # Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                # Read feed
                ret, frame = cap.read()
                if not ret or frame is None:
                    print("No frame captured")
                    continue

                # Make detections
                image, results = mediapipe_detection(frame, holistic)

                # Draw landmarks
                draw_styled_landmarks(image, results)
                
                # NEW Apply wait logic
                if frame_num == 0: 
                    cv2.putText(image, 'STARTING COLLECTION', (120,200), 
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    
                else: 
                    cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                
                # Show to screen - force window creation
                cv2.namedWindow('OpenCV Feed', cv2.WINDOW_AUTOSIZE)
                cv2.imshow('OpenCV Feed', image)
                
                # Force window update
                key = cv2.waitKey(30) & 0xFF
                if key == ord('q'):
                    break
                
                # NEW Export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

# Cleanup outside 'with' block
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
    
# cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
# # Set mediapipe model 
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
#     # NEW LOOP
#     # Loop through actions
#     for action in actions:
#         for sequence in range(no_sequences):
#             try:
#                 os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
#             except:
#                 pass
#             # Loop through sequences aka videos
#             for sequence in range(no_sequences):
#                 cv2.waitKey(0)
                
#                 # Loop through video length aka sequence length
#                 for frame_num in range(sequence_length):

#                     # Read feed
#                     ret, frame = cap.read()

                    # Make detections
#                     image, results = mediapipe_detection(frame, holistic)
#                     # print(results)

#                     # Draw landmarks
#                     draw_styled_landmarks(image, results)
                    
#                     # NEW Apply wait logic
#                     if frame_num == 0: 
#                         cv2.putText(image, 'STARTING COLLECTION', (120,200), 
#                                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
#                         cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
#                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                         # Show to screen
#                         cv2.imshow('OpenCV Feed', image)
#                         cv2.waitKey(1000)
#                     else: 
#                         cv2.putText(image, 'Collecting frames for {} Video Number {}'.format(action, sequence), (15,12), 
#                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
#                         # Show to screen
#                         cv2.imshow('OpenCV Feed', image)
                    
#                     # NEW Export keypoints
#                     keypoints = extract_keypoints(results)
#                     npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
#                     np.save(npy_path, keypoints)

#                     # Break gracefully
#                     if cv2.waitKey(10) & 0xFF == ord('q'):
#                         break
                    
# cap.release()
# cv2.destroyAllWindows()

In [ ]:
cap.release()
cv2.destroyAllWindows()

# 6. Preprocess Data and Create Labels and Features

In [22]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [23]:
label_map = {label:num for num, label in enumerate(actions)}

In [26]:
label_map

{'Vsih vitay': 0, 'Diakuy vam': 1}

In [28]:
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [30]:
np.array(sequences).shape

(100, 30, 126)

In [32]:
np.array(labels).shape

(100,)

In [34]:
X = np.array(sequences)

In [36]:
X.shape

(100, 30, 126)

In [38]:
y = to_categorical(labels).astype(int)

In [40]:
y

array([[1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [0,

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

In [44]:
y_test.shape

(5, 2)

In [46]:
X_train.shape

(95, 30, 126)

# 7. Build and Train LSTM Neural Network

In [48]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [50]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [52]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 126)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [54]:
res = [.7, 0.2, 0.1]

In [56]:
actions[np.argmax(res)]

'Vsih vitay'

In [58]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [60]:
model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback])

Epoch 1/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - categorical_accuracy: 0.5366 - loss: 0.6932
Epoch 2/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - categorical_accuracy: 0.5183 - loss: 0.6930
Epoch 3/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - categorical_accuracy: 0.4245 - loss: 0.6948
Epoch 4/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - categorical_accuracy: 0.5065 - loss: 0.6929
Epoch 5/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - categorical_accuracy: 0.4948 - loss: 0.6927
Epoch 6/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - categorical_accuracy: 0.4792 - loss: 0.6931
Epoch 7/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - categorical_accuracy: 0.4870 - loss: 0.6917
Epoch 8/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - categorical_accuracy: 0.5534 - loss: 0.6880
Epoch 9/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - categorical_accuracy: 0.4792 - loss: 0.6869
Epoch 10/2000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - categorical_accuracy: 0.4987 - loss: 0.6773
Epoch 11/2000
3/3 ━━━━━━━━━━━

In [62]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        48,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 610,280 (2.33 MB)

 Trainable params: 203,426 (794.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 406,854 (1.55 MB)

# 8. Make Predictions

In [65]:
res = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 603ms/step


In [67]:
actions[np.argmax(res[4])]

'Diakuy vam'

In [69]:
actions[np.argmax(y_test[4])]

'Vsih vitay'

# 9. Save Weights

In [72]:
model.save('action.h5')

In [74]:
model.load_weights('action.h5')

# 10. Evaluation using Confusion Matrix and Accuracy

In [77]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [79]:
yhat = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


In [81]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [83]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[0, 2],
        [1, 2]],

       [[2, 1],
        [2, 0]]])

In [85]:
accuracy_score(ytrue, yhat)

0.4

# 11. Test in Real Time

In [ ]:
from PIL import ImageFont, ImageDraw, Image
import platform
#import pyttsx3

sequence = []
sentence = ["Press R to start"]
is_recording = False
threshold = 0.8

# Fix font path for macOS
try:
    if platform.system() == "Darwin":  # macOS
        font = ImageFont.truetype("/System/Library/Fonts/Arial.ttf", 35)
    elif platform.system() == "Windows":
        font = ImageFont.truetype(r"C:\Windows\Fonts\arial.ttf", 35)
    else:  # Linux
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 35)
except OSError:
    font = ImageFont.load_default()

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        if cv2.waitKey(1) & 0xFF == ord('r'):
            if not is_recording:
                is_recording = True
                sequence = []
                sentence = [""]

        if is_recording:
            # 2. Prediction logic
            keypoints = extract_keypoints(results)
            sequence.append(keypoints)
            sequence = sequence[-30:]  # Fixed syntax error
        
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(sequence)
            print(res)
            is_recording = False
            sequence = []
            print(actions[np.argmax(res)])
            
            if res[np.argmax(res)] > threshold:
                sentence[0] = actions[np.argmax(res)]
                #engine = pyttsx3.init()
                #engine.say(sentence[0])
                # play the speech
                #engine.runAndWait()
            else:
                sentence[0] = "Try again"

        # Convert BGR to RGB for PIL
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(image_rgb)
        draw = ImageDraw.Draw(pil_image)
        draw.text((30, 30), sentence[0], font=font, fill=(255, 255, 255))
        # Convert back to BGR for OpenCV
        image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# from PIL import ImageFont, ImageDraw, Image
# #import pyttsx3

# sequence = []
# sentence = ["Press R to start"]
# is_recording = False
# threshold = 0.8

# cap = cv2.VideoCapture(0)
# # Set mediapipe model 
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
#     while cap.isOpened():

#         # Read feed
#         ret, frame = cap.read()

#         # Make detections
#         image, results = mediapipe_detection(frame, holistic)
        
#         # Draw landmarks
#         draw_styled_landmarks(image, results)
#         print
#         if cv2.waitKey(1) & 0xFF == ord('r'):
#             if not is_recording:
#                 is_recording = True
#                 sequence = []
#                 sentence = [""]

#         if is_recording:
#             # 2. Prediction logic
#             keypoints = extract_keypoints(results)
    #         sequence.insert(0,keypoints)
    #         sequence = sequence[:30]
    #         sequence.append(keypoints)
    #         sequence = sequence[-30:]
        
    #     if len(sequence) == 30:
    #         res = model.predict(np.expand_dims(sequence, axis=0))[0]
    #         print(sequence)
    #         print(res)
    #         is_recording = False
    #         sequence = []
    #         print(actions[np.argmax(res)])
            
    #         if res[np.argmax(res)] > threshold:
    #             sentence[0] = actions[np.argmax(res)]
    #             #engine = pyttsx3.init()
    #             #engine.say(sentence[0])
    #             # play the speech
    #             #engine.runAndWait()
    #         else:
    #             sentence[0] = "Try again"

    #     pil_image = Image.fromarray(image)
    #     font = ImageFont.truetype("C:\Windows\Fonts\\arial.ttf", 35)
    #     draw = ImageDraw.Draw(pil_image)
    #     draw.text((30, 30), sentence[0], font=font)
    #     image = np.asarray(pil_image)

    #     #cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        
    #     # Show to screen
    #     cv2.imshow('OpenCV Feed', image)

    #     # Break gracefully
    #     if cv2.waitKey(10) & 0xFF == ord('q'):
    #         break
    # cap.release()
    # cv2.destroyAllWindows()

I0000 00:00:1753872640.349478 1398951 gl_context.cc:369] GL version: 2.1 (2.1 INTEL-22.5.10), renderer: Intel(R) Iris(TM) Plus Graphics 645
W0000 00:00:1753872640.594591 1491925 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753872640.659751 1491931 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753872640.670935 1491929 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753872640.672320 1491927 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1753872640.672407 1491926 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature i

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 549ms/step
[array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0.]), array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 